# Tutorial: advanced functionality
This notebook aims to show some advanced functionality of the code-base that is outside of the scope of the general tutorial notebook.

## Custom data-type convolution
here we explain how to implement hooks to convolve custom data-types that is not dataframe based event-type data or nested-dict ensemble-type data.  

TODO

## Post-convolution processing function
Here we explain how to implement a post-convolution processing function call. This can be used for a range of reasons, e.g. filtering of the data to only include particular systems, providing additional detection-probability weighting to go from intrinsic rates to predicted observed rates, further evolution of sampled systems to the present-day through gravitational-wave radiation, orbit integration (including kicks), etc. 


The user can provide the `convolution_instructions` dictionary with an additional function that performs some additional calculation. This is done as follows. The extra_weights_function should be stored in the `convolution_instructions` as `convolution_instructions['post_convolution_function']`. This function should return an array of additional weights.

The arguments to this function do not require any specific parameter, but the convolution code internally allows the following arguments to be passed to the function:

- `config`: general configuration dictionary
- `time_value`: the time value of the current time-bin (can be redshift as well, but must be named `time_value`.
- `convolution_instruction`: convolution instruction dictionary.
- `job_dict`: dictionary containing current job information
- `sfr_dict`: dictionary containing the star formation rate dictionary
- `result_dict`: dictionary containing the results of the current convolution step. This one is rather important. TODO: likely is actually required.
- `data_dict`: dictionary that contains the data. This will be filled with the keys and data that either `data_layer_dict` or `data_column_dict` instructs the convolution code to use.
- `**convolution_instruction["post_convolution_function_extra_parameters"]`: additional parameters and values that the user can provide.

Which of these arguments gets passed depends on the arguments to the `post_convolution_function`, which are automatically recognized and passed to the function.

This function should return an updated `data_dict` dictionary, where the results are updated accordingly. Apart from updating the `yield` field, which contains the conventional convolution results, one (in some cases, see below) can add extra fields to this dictionary, like its position in space.

Depending on the type of convolution, and the data type, the shape (length) of the entries in this dictionary can have different requirements. 

This is to maintain correct data-integrity.
- event convolution by integration: length of arrays has to be the same before as after
- event convolution by sampling: no restrictions
- ensemble convolution by integration: length of arrays has to be the same as before, and no additional fields can be added (yet).

An example is as follows:

```python
def post_convolution_function_detection_probability(config, time_value, data_dict, min_detection_probability):
    # just an example, this is not how detection probability is calculated
    detection_probability = data_dict['mass_1'] * data_dict['mass_2'] * time_value
    
    detection_probability[detection_probability<min_detection_probability] = 0

    return detection_probability

# 
convolution_config['convolution_instructions'] = [
    {
        'input_data_type': 'event',
        'input_data_name': 'RLOF',
        'output_data_name': 'RLOF_rate',
        'data_column_dict': {
            # required
            'delay_time': 'initial_time',
            'yield_rate': 'probability',
            # optional*
            'metallicity': 'metallicity',
            # optional**
            'mass_1': 'RLOF_initial_mass_accretor',
            'mass_2': 'RLOF_initial_mass_donor',
        },
        'post_convolution_function': post_convolution_function_detection_probability,
        'post_convolution_function_extra_parameters': {'min_detection_probability': 1e-6}
    },
]

```

PS: the use of `extra_weights_function_additional_parameters` is somewhat unnecessary, as one can either hardcode the additional parameters within the function, or make use of the `partial` method (see https://www.geeksforgeeks.org/partial-functions-python/)

Some useful resources:
- https://arxiv.org/abs/2404.16930 for GW detection
- https://gaiaunlimited.readthedocs.io/ for gaia selection functions



## advanced convolution-instructions 

- `conversion_factor`: (optional) factor by which the data is multiplied
- `conversion_function`: (optional) function that is applied to the data. This function should only accept 1 argument, which is the data.

An additional layer-list can be provided to marginalise, or remove certain layers from the output ensemble. This list is called `marginalisation_list`

As an example:

```python
convolution_config['convolution_instructions'] = [
    {
        'input_data_type': 'ensemble',
        'input_data_name': 'RLOF',
        'output_data_name': 'RLOF_rate',
        'data_layer_dict': {
            # required
            'delay_time': 5,
            # optional*
            'metallicity': 1,
            # optional**
            'mass_1': 2,
            'mass_2': 3,
        },
        'marginalisation_list': [2,3]
    },
]
```